<div align="center">
  <h1>QPE for Fe<sub>2</sub>S<sub>2</sub>: simulation, resources and GPU speedup</h1>
  <p>
    <img src="https://img.shields.io/badge/Python-3.11-3776AB?style=flat&logo=python" alt="Python">
    <img src="https://img.shields.io/badge/CUDA-76B900?style=flat&logo=nvidia" alt="CUDA-Q">
  </p>
  <i>This demo is part of a project between <a href="https://quantummotion.com/">Quantum Motion</a> and NVIDIA to <a href="https://quantummotion.com/state-preparation-for-quantum-chemistry-with-quantum-motion-and-nvidia-cuda-q">improve the state preparation pipeline for quantum chemistry</a>.</i>
</div>

---

This demo takes the [Fe<sub>2</sub>S<sub>2</sub>(SCH<sub>3</sub>)<sub>4</sub>]<sup>2&minus;</sup> cluster through the full QUICHE pipeline
(ORCA &rarr; QUICHE / CUDA-Q Hamiltonian &rarr; DMRG guide state &rarr; QPE) and does three things:

1. **Simulate** QPE for the minimal CAS(10e,10o) active space (20 system qubits).
2. **Compare the resources** QPE needs for three active spaces: CAS(10e,10o), CAS(30e,20o) and CAS(30e,32o).
3. **Compare CPU and GPU** runtimes for the steps that dominate the cost.

### Hardware

<div style="background-color: #fff3cd; color: #856404; border-left: 5px solid #ffeeba; padding: 15px; margin-top: 15px; margin-bottom: 15px; border-radius: 4px;">
  <strong>⚠️ Hardware Notice:</strong> CUDA-Q kernels run on a GPU when one is detected and fall back to <code>qpp-cpu</code> otherwise.
  On CPU, runtime grows quickly with the number of phase qubits, so the simulation limit (<code>max_qubits</code>) is lower. The CPU-vs-GPU comparison in Part 3 needs a GPU to show a speedup.
</div>

## Configuration

Everything you may want to change is in the next cell. `n_cores` is used both by ORCA and by the CPU simulator, so the CPU baseline in the speedup comparison is a fair multi-core one.

In [ ]:
# Paths
orca_dir = "~/Software/orca_6_1_1_macosx_arm64_openmpi411"

# Parallelism: used by ORCA and by CUDA-Q's CPU backend (qpp-cpu, OpenMP).
n_cores = 8

# QPE error budgets (Ha) and shots per run.
errors = [2.0, 1.0, 0.5, 0.25, 0.125, 0.0625, 0.03125, 0.015625]
shots = 1000
CHEM_ACC = 1.6e-3  # Ha

# Statevector limits (total qubits = system + phase qubits).
max_qubits_gpu, max_qubits_cpu = 33, 26  # one 80 GB GPU in fp32 / a typical CPU node

# True: rerun ORCA. False: reuse the ORCA output and FCIDUMP shipped with the demo.
recompute_orca = True

### Imports

The import block is collapsed. It also applies `n_cores` to the CPU simulator, which has to happen before CUDA-Q is imported.

In [ ]:
# System imports.
import contextlib
import functools
import os
import time
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Thread count for CUDA-Q's CPU backend; must be set before cudaq is imported.
os.environ["OMP_NUM_THREADS"] = str(n_cores)
os.environ["PATH"] = os.path.expanduser(orca_dir) + os.pathsep + os.environ["PATH"]

# Orca imports.
from opi.core import Calculator
from opi.input.simple_keywords import BasisSet, Scf, ShellType
from opi.input.structures import Structure
from opi.input.blocks import BlockCasscf, BlockOutput

# CudaQ imports (before quiche: cudaq must be the first native extension loaded).
from cudaq_algorithms import chemistry
import cudaq

# Quiche imports.
import quiche
from quiche.chemistry import HartreeFockState, get_jw_state
from quiche.core import ElectronicHamiltonian, Errors, PhaseEstimation, Simulation
from quiche.cudaq import measured_circuit_kernel, sample_post_processor, textbook_qpe_circuit
from quiche.dispatch import QPESpec
from quiche.io import fcidump

# In-house helper imports.
from statePrep.cudaq_circuit_loading import load_circuit_from_json, write_and_import_kernel

### CUDA-Q backend

The same helper switches between backends later, when we compare CPU and GPU.

In [ ]:
has_gpu = cudaq.num_available_gpus() > 0
max_qubits = max_qubits_gpu if has_gpu else max_qubits_cpu

def set_backend(target, option=None):
    cudaq.set_target(target, option=option) if option else cudaq.set_target(target)

TARGET = ("nvidia", None) if has_gpu else ("qpp-cpu", None)
set_backend(*TARGET)
print("CUDA-Q target:", TARGET[0], "| GPUs:", cudaq.num_available_gpus(), "| CPU cores:", n_cores)

## The system: three Fe<sub>2</sub>S<sub>2</sub> active spaces

[Fe<sub>2</sub>S<sub>2</sub>(SCH<sub>3</sub>)<sub>4</sub>]<sup>2&minus;</sup> (charge &minus;2, singlet) is a strongly correlated iron&ndash;sulfur cluster and a stepping stone towards FeMoco. We use three active spaces:

| Active space | Orbitals | System qubits |
| --- | --- | --- |
| CAS(10e,10o) | Fe 3d | 20 |
| CAS(30e,20o) | Fe 3d, bridging S 3p, one 3p per terminal S | 40 |
| CAS(30e,32o) | as above, plus Fe 4s and 4d (double shell) | 64 |

The active orbitals were selected beforehand with ORCA's **AVAS** method, which rotates converged RHF orbitals onto the requested atomic orbitals. That selection is not repeated here: the AVAS orbitals are shipped with the demo as one `.gbw` file per active space, and ORCA starts its CASSCF from them. We use def2-SVP because STO-3G has no Fe 4d functions.

For the (10e,10o) space the CASSCF is exact, so QPE errors are measured against the ORCA energy. The larger spaces use ORCA's approximate ICE-CI, so there DMRG is the reference.

In [ ]:
@dataclass(frozen=True)
class System:
    """One molecule in one active space. Files live in <molecule>/<nel>e<norb>o/."""

    nel: int
    norb: int
    molecule: str = "Fe2S2"
    charge: int = -2
    multiplicity: int = 1
    basis: BasisSet = BasisSet.DEF2_SVP
    casscf_options: dict = field(default_factory=dict)  # extra %casscf settings
    reference: str = "casscf"  # energy QPE errors are measured against: "casscf" or "dmrg"

    @property
    def name(self) -> str:
        return f"{self.molecule} ({self.nel}e,{self.norb}o)"

    @property
    def workdir(self) -> Path:
        return Path.cwd() / self.molecule / f"{self.nel}e{self.norb}o"

    @property
    def fcidump_path(self) -> Path:
        return self.workdir / f"{self.molecule}.fcidump"

    @property
    def num_qubits(self) -> int:
        return 2 * self.norb


ice = dict(casscf_options={"cistep": "ice"}, reference="dmrg")
SYSTEMS = {s.name: s for s in [System(10, 10), System(30, 20, **ice), System(30, 32, **ice)]}
demo = SYSTEMS["Fe2S2 (10e,10o)"]
list(SYSTEMS)

# Part 1: Simulate QPE for CAS(10e,10o)

## Step 1: Run ORCA

ORCA runs a CASSCF in the active space, starting from the shipped AVAS orbitals, and dumps the active-space integrals as an FCIDUMP. Set `recompute_orca = False` to skip the calculation and read the shipped output instead.

In [ ]:
def run_orca(system: System) -> float:
    """Run CASSCF from the AVAS orbitals; write the FCIDUMP and return the CASSCF energy."""
    wd = system.workdir
    calc = Calculator(basename=system.molecule, working_dir=wd)
    calc.structure = Structure.from_xyz(wd.parent / f"{system.molecule}.xyz")
    calc.structure.charge, calc.structure.multiplicity = system.charge, system.multiplicity
    calc.input.ncores = n_cores
    calc.input.add_simple_keywords(ShellType.RHF, system.basis, Scf.MOREAD)
    calc.input.moinp = wd / f"{system.molecule}_avas.gbw"
    calc.input.add_blocks(
        BlockCasscf(nel=system.nel, norb=system.norb, mult=system.multiplicity, **system.casscf_options)
    )
    dump = BlockOutput()
    dump.add_option("DumpActInts", "True")  # writes <molecule>.fcidump
    calc.input.add_blocks(dump)

    if recompute_orca:
        calc.write_input()
        calc.run()
    output = calc.get_output()
    if not output.terminated_normally():
        raise RuntimeError(f"ORCA did not terminate normally for {system.name}.")
    output.parse()
    return output.get_final_energy()


t0 = time.perf_counter()
e_orca = run_orca(demo)
print(f"{demo.name}: CASSCF energy = {e_orca:.10f} Eh  ({time.perf_counter() - t0:.0f} s on {n_cores} cores)")

## Step 2: Hamiltonian in QUICHE via CUDA-Q Algorithms

QUICHE parses the FCIDUMP, CUDA-Q Algorithms builds the Jordan&ndash;Wigner qubit Hamiltonian, and we convert it to a QUICHE `PauliSum`.

In [ ]:
@functools.cache
def hamiltonian(name: str):
    """Parse the system's FCIDUMP; return (fermionic Hamiltonian, CUDA-Q operator, QUICHE PauliSum)."""
    sq = fcidump.read(str(SYSTEMS[name].fcidump_path))
    h_cq = chemistry.qubit_hamiltonian(sq.one_body, sq.two_body, scalar_offset=sq.core_energy)
    return sq, h_cq, quiche.PauliSum.from_cudaq(h_cq)


sq_ham, h_cq, h = hamiltonian(demo.name)
print(f"{demo.name}: {h.num_qubits} qubits, {h.num_words} Pauli terms")

## Step 3: DMRG guide state

The guide state was prepared with DMRG (MPSCircuits.jl) and compiled into a circuit, as in Part 1 of the tutorial series. Here we load that circuit as a CUDA-Q kernel and check it against the reference energy and its overlap with Hartree&ndash;Fock. The overlap with Hartree&ndash;Fock is small for this strongly multireference cluster, which is why a good guide state matters.

In [ ]:
def load_dmrg(system: System):
    """Load the DMRG circuit as a CUDA-Q kernel, plus the DMRG summary (energy, ...)."""
    info_path = system.workdir / f"{system.molecule}_dmrg_info.txt"
    circuit_path = system.workdir / f"{system.molecule}_dmrg_circuit.json"
    info = dict(line.split("=") for line in info_path.read_text().splitlines() if "=" in line)
    # Write the generated kernel into the system's own folder.
    with contextlib.chdir(system.workdir):
        tag = f"dmrg_{system.nel}e{system.norb}o"
        kernel = write_and_import_kernel(load_circuit_from_json(circuit_path), kernel_name=tag, module_name=tag)
    return kernel, info


dmrg, dmrg_info = load_dmrg(demo)

n = demo.num_qubits

@cudaq.kernel
def dmrg_state():
    q = cudaq.qvector(n)
    dmrg(q)

hf = HartreeFockState.closed_shell(num_electrons=sq_ham.num_electrons, num_spin_orbitals=n)
hf_bits = "".join(str(int(b)) for b in get_jw_state(hf.occupation))  # Jordan-Wigner, qubit i = character i

print(f"ORCA CASSCF energy (reference): {e_orca:.8f} Ha")
print(f"DMRG energy (Julia):            {float(dmrg_info['dmrg_energy']):.8f} Ha")
print(f"DMRG energy (CUDA-Q circuit):   {cudaq.observe(dmrg_state, h_cq).expectation():.8f} Ha")
print(f"HF overlap |<HF|DMRG>|^2:       {abs(cudaq.get_state(dmrg_state).amplitude(hf_bits)) ** 2:.4f}")

## Step 4: QPE from an error budget

Instead of choosing the number of phase qubits and Trotter steps by hand, we specify a **target energy error** and QUICHE's `QPESpec` resolves the rest: the number of phase qubits, the Trotter time, order and number of steps. The same spec is then lowered to a CUDA-Q circuit.

In [ ]:
def get_qpe(sq, h, error: float) -> QPESpec:
    """Size the QPE for an error budget (the DMRG kernel is injected when building the circuit)."""
    return QPESpec(
        hamiltonian=ElectronicHamiltonian(
            num_electrons=sq.num_electrons, mapping=quiche.Mapping.JordanWigner, paulis=h
        ),
        state_prep=None,
        algorithm=PhaseEstimation.Textbook,
        simulation=Simulation.Trotter,
        error_budget=Errors(
            estimation=error / 2, simulation=error / 2, rotations=error / 2, state_prep=error / 2, overlap=1.0
        ),
    )


def qpe_circuit(qpe, h, dmrg, num_ancillas=None, reps=None):
    """Lower the spec to a sampled CUDA-Q circuit; returns (circuit, post-processor)."""
    ancillas = num_ancillas or qpe.num_qpe_ancillas
    kernel, work_qubits, decode = textbook_qpe_circuit(
        h, Simulation.Trotter, ancillas, qpe.time, reps or qpe.reps, qpe.order, n_qubits=qpe.num_data
    )
    circuit = measured_circuit_kernel(kernel, work_qubits, ancillas, state_prep=dmrg, num_data=qpe.num_data)
    return circuit, sample_post_processor(decode, 1)


def describe(qpe) -> dict:
    return dict(
        total_qubits=qpe.num_data + qpe.num_qpe_ancillas,
        phase_qubits=qpe.num_qpe_ancillas,
        trotter_reps=qpe.reps,
        # Controlled Trotter-circuit applications (2^m - 1), each with `reps` steps.
        trotter_steps=(2**qpe.num_qpe_ancillas - 1) * qpe.reps,
    )


def time_sample(circuit, shots, repeats=1):
    """Sample the circuit; return (samples, median wall time). A 1-shot warm-up excludes JIT compilation."""
    cudaq.sample(circuit, shots_count=1)
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        samples = cudaq.sample(circuit, shots_count=shots)
        times.append(time.perf_counter() - t0)
    return samples, float(np.median(times))


pd.DataFrame({error: describe(get_qpe(sq_ham, h, error)) for error in errors}).T.rename_axis("error_budget")

## Step 5: Accuracy versus cost

For each error budget we build the circuit, simulate it on the active backend and record the median QPE energy, the error against the reference and the wall time. Budgets that need more than `max_qubits` qubits are listed in the table above but not simulated.

In [ ]:
def error_sweep(errors, reference, repeats=1) -> pd.DataFrame:
    rows = []
    for error in errors:
        qpe = get_qpe(sq_ham, h, error)
        info = describe(qpe)
        if info["total_qubits"] > max_qubits:
            print(f"error {error}: needs {info['total_qubits']} qubits, skipped")
            continue
        circuit, post = qpe_circuit(qpe, h, dmrg)
        samples, wall = time_sample(circuit, shots, repeats)
        energy = float(np.median(post(samples)))
        rows.append(dict(error_budget=error, energy=energy, abs_error=abs(energy - reference), wall_s=wall, **info))
        print(f"error {error}: |E - E_ref| = {rows[-1]['abs_error']:.2e} Ha in {wall:.2f} s")
    return pd.DataFrame(rows)


df_sweep = error_sweep(errors, reference=e_orca)
df_sweep

In [ ]:
def plot_accuracy(df, label):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    axes[0].loglog(df.wall_s, df.abs_error, "o-", label=label)
    axes[1].loglog(df.trotter_steps, df.abs_error, "o-", label=label)
    for ax, xlabel in zip(axes, ["Simulated wall time (s)", "Total Trotter steps"]):
        ax.axhline(CHEM_ACC, color="tab:green", ls=":", label="Chemical accuracy")
        ax.set_xlabel(xlabel)
    axes[0].set_ylabel("|E_QPE − E_ref| (Ha)")
    axes[0].legend()
    plt.tight_layout()


plot_accuracy(df_sweep, demo.name)

# Part 2: Resources for three active spaces

Simulation stops at CAS(10e,10o): 20 system qubits plus the phase register already fills a GPU, and CAS(30e,32o) needs 64 system qubits. The same QPE specification, however, can be sized for any active space. Here we compare, at chemical accuracy ($1.6\,\mathrm{mHa}$), how large the Pauli-Trotter QPE gets as the active space grows. This only needs the FCIDUMPs, not a simulation.

In [ ]:
def resource_table(names, error=CHEM_ACC) -> pd.DataFrame:
    rows = []
    for name in names:
        sq, _, h_sys = hamiltonian(name)
        qpe = get_qpe(sq, h_sys, error)
        rows.append(dict(system=name, system_qubits=qpe.num_data, pauli_terms=h_sys.num_words, **describe(qpe)))
    return pd.DataFrame(rows).set_index("system")


df_res = resource_table(SYSTEMS)
df_res

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, col, label in zip(
    axes,
    ["pauli_terms", "total_qubits", "trotter_steps"],
    ["Pauli terms", "Total qubits (system + phase)", "Total Trotter steps"],
):
    ax.bar(range(len(df_res)), df_res[col], color=["tab:blue", "tab:orange", "tab:red"])
    ax.set_xticks(range(len(df_res)), [n.split()[-1] for n in df_res.index])
    ax.set_yscale("log")
    ax.set_ylabel(label)
plt.tight_layout()

### Logical resources with DFTHC qubitisation

<div style="background-color: #d1ecf1; color: #0c5460; border-left: 5px solid #bee5eb; padding: 15px; margin-top: 15px; margin-bottom: 15px; border-radius: 4px;">
  <strong>ℹ️ To be added:</strong> Toffoli and logical-qubit counts from DFTHC qubitisation. They need a DFTHC factorisation of each active space, which comes from an external code and is not wired into this demo yet. The table above is laid out so those columns can be appended per system.
</div>

The cell below is where that step will go. It is also the second step, after the simulation, that we compare on CPU and GPU in Part 3.

In [ ]:
def dfthc_factorisation(system: System, device: str = "cpu"):
    """Placeholder: run the external DFTHC factorisation of `system` on `device` ("cpu" or "gpu")."""
    raise NotImplementedError("Hook up the external DFTHC factorisation here.")

# Part 3: CPU versus GPU

We compare CPU and GPU on the two steps that dominate the cost: the QPE **simulation** (this section) and the **DFTHC factorisation** used for the resource estimate (placeholder below).

## QPE simulation

Following Part 2 of the tutorial series, we time the same QPE circuit on each backend while increasing the number of phase qubits. The number of Trotter steps per controlled application is fixed at 1, so only the circuit width and the backend change. The CPU baseline uses `n_cores` threads. `fp64` is included because QPE at mHa resolution applies very long gate sequences, where single-precision rounding can accumulate.

In [ ]:
BACKENDS = {
    f"CPU ({n_cores} cores)": ("qpp-cpu", None),
    "GPU fp32": ("nvidia", None),
    "GPU fp64": ("nvidia", "fp64"),
}
CPU = next(iter(BACKENDS))


def backend_benchmark(phase_qubits, shots=100, repeats=2) -> pd.DataFrame:
    qpe = get_qpe(sq_ham, h, error=1.0)  # only provides time, order and the data-qubit count
    rows = []
    for label, (target, option) in BACKENDS.items():
        if target != "qpp-cpu" and not has_gpu:
            continue
        limit = max_qubits_cpu if target == "qpp-cpu" else max_qubits_gpu
        set_backend(target, option)  # circuits are rebuilt after every switch
        for m in phase_qubits:
            if qpe.num_data + m > limit:
                continue
            circuit, _ = qpe_circuit(qpe, h, dmrg, num_ancillas=m, reps=1)
            _, wall = time_sample(circuit, shots, repeats)
            rows.append(dict(backend=label, total_qubits=qpe.num_data + m, wall_s=wall))
            print(f"{label}: {qpe.num_data + m} qubits in {wall:.2f} s")
    set_backend(*TARGET)
    return pd.DataFrame(rows)


def speedup_table(df, x, baseline):
    """Wall times and speedups (baseline time / time) per backend, indexed by `x`."""
    wall = df.pivot(index=x, columns="backend", values="wall_s")
    return wall, wall.drop(columns=baseline).rdiv(wall[baseline], axis=0)


def plot_speedup(wall, speedup, xlabel):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for col in wall:
        axes[0].semilogy(wall.index, wall[col], "o-", label=col)
    for col in speedup:
        axes[1].plot(speedup.index, speedup[col], "o-", label=col)
    axes[1].axhline(1, color="grey", ls=":")
    axes[0].set_ylabel("Wall time (s)")
    axes[1].set_ylabel(f"Speedup over {CPU}")
    for ax in axes:
        ax.set_xlabel(xlabel)
        ax.legend()
    plt.tight_layout()


df_bench = backend_benchmark(phase_qubits=[2, 4, 6, 8, 10, 12])
wall, speedup = speedup_table(df_bench, "total_qubits", CPU)
plot_speedup(wall, speedup, "Total qubits")
speedup.round(1)

## DFTHC factorisation

The same comparison for the factorisation step reuses `speedup_table` and `plot_speedup`: time `dfthc_factorisation(system, device)` for each device and active space, collect the times in a table with columns `backend`, `orbitals` and `wall_s`, and plot them. This runs once the external factorisation code is hooked in above.

In [ ]:
# Once dfthc_factorisation is implemented:
#
# rows = []
# for name, system in SYSTEMS.items():
#     for device, label in [("cpu", CPU), ("gpu", "GPU")]:
#         t0 = time.perf_counter()
#         dfthc_factorisation(system, device)
#         rows.append(dict(backend=label, orbitals=system.norb, wall_s=time.perf_counter() - t0))
# wall, speedup = speedup_table(pd.DataFrame(rows), "orbitals", CPU)
# plot_speedup(wall, speedup, "Active orbitals")

---

<div align="center">
  <h2>🎉 Demo Complete!</h2>
</div>

🏆 **What you've accomplished:**

- Ran ORCA and QPE for the minimal Fe<sub>2</sub>S<sub>2</sub> active space, CAS(10e,10o), and measured accuracy against cost
- Compared how large QPE gets for CAS(10e,10o), CAS(30e,20o) and CAS(30e,32o) at chemical accuracy
- Timed the same QPE circuit on CPU and GPU and measured the speedup

---